# 4.03 Grid Search de Arboles Azarosos — Opcion A: feedback via Kaggle

Recorre la grilla de hiperparametros < feature_fraction, minsplit, minbucket, maxdepth > (con **cp fijo en -1**), entrenando para cada combinacion el ensemble arbol por arbol (igual que en `z420`) y subiendo un submit a Kaggle en cada punto de `PARAM$grabar` (por default `1, 2, 4, 8, 16, 32`), con el detalle de la combinacion y el arbolito en el mensaje del submit.

<br>**Importante**: cada combinacion consume `length(PARAM$grabar)` submits. El limite diario de submits es 100 (segun indico el profesor) — hay una celda que calcula cuantos submits va a consumir la grilla actual antes de correrla.

<br>Una vez que tengas los puntajes del Public Leaderboard para cada combinacion y arbolito (mensaje del submit en Kaggle > My Submissions), elegis la mejor y la cargas manualmente en `z420` (los campos `PARAM$feature_fraction` y `PARAM$rpart`) para el entrenamiento final.

#### Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"

---

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Aqui debe cargar SU semilla primigenia, y ajustar la grilla de hiperparametros a explorar

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 346321

PARAM$rpart$cp <- -1 # fijo, no se barre en este grid search

PARAM$num_trees_max <- 32 # arboles por combinacion, igual que z420

# puntos del ensemble donde se sube un submit a Kaggle, igual que en z420
PARAM$grabar <- c(1, 2, 4, 8, 16, 32)

# grilla de hiperparametros a explorar
# OJO: cada combinacion de esta grilla genera length(PARAM$grabar) submits a Kaggle
#  el limite diario de submits es 100 (segun indico el profesor)
PARAM$grid$feature_fraction <- c(0.3, 0.5, 0.7)
PARAM$grid$minsplit  <- c(1000)
PARAM$grid$minbucket <- c(100)
PARAM$grid$maxdepth  <- c(6, 8)

In [ ]:
PARAM

In [ ]:
# chequeo cuantos submits va a consumir esta grilla, contra el limite diario de 100
qty_combos <- length(PARAM$grid$feature_fraction) * length(PARAM$grid$minsplit) *
  length(PARAM$grid$minbucket) * length(PARAM$grid$maxdepth)
qty_submits_planificados <- qty_combos * length(PARAM$grabar)

message("combinaciones (sin filtrar minbucket>minsplit): ", qty_combos)
message("submits planificados: ", qty_submits_planificados, " / 100 por dia")

if (qty_submits_planificados > 100) {
  warning("La grilla actual planifica mas submits que el limite diario, achicala o correla en varios dias")
}

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp4211"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

# arreglo clase_ternaria por algun distraido ""
dfuture[, clase_ternaria := NA ]

In [ ]:
# Establezco cuales son los campos que puedo usar para la prediccion
campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria")))

### Grid Search

In [ ]:
# archivo donde se guarda el checkpoint del grid search (que combinaciones+arbolito ya se subieron)
archivo_grid <- "gridsearch_kaggle.txt"

if (file.exists(archivo_grid)) {
  tb_grid <- fread(archivo_grid)
} else {
  tb_grid <- data.table(
    combo_id = integer(),
    feature_fraction = numeric(),
    cp = numeric(),
    minsplit = integer(),
    minbucket = integer(),
    maxdepth = integer(),
    arbolito = integer(),
    archivo_kaggle = character()
  )
}

combo_id <- 0

for (v_feature_fraction in PARAM$grid$feature_fraction) {
for (v_minsplit in PARAM$grid$minsplit) {
for (v_minbucket in PARAM$grid$minbucket) {
for (v_maxdepth in PARAM$grid$maxdepth) {

  if (v_minbucket > v_minsplit) next # combinacion redundante/invalida

  combo_id <- combo_id + 1

  # si esta combinacion ya tiene TODOS los puntos de grabar subidos, me la salteo entera
  puntos_hechos <- tb_grid[
    feature_fraction == v_feature_fraction &
    minsplit == v_minsplit &
    minbucket == v_minbucket &
    maxdepth == v_maxdepth,
    arbolito
  ]
  if (all(PARAM$grabar %in% puntos_hechos)) next

  message("combo ", combo_id, " : feature_fraction=", v_feature_fraction,
    " minsplit=", v_minsplit, " minbucket=", v_minbucket, " maxdepth=", v_maxdepth)

  rpart_control <- list(
    cp= PARAM$rpart$cp,
    minsplit= v_minsplit,
    minbucket= v_minbucket,
    maxdepth= v_maxdepth
  )

  # misma semilla para todas las combinaciones, asi la unica diferencia entre
  #  combinaciones es el hiperparametro, no el azar
  set.seed(PARAM$semilla_primigenia)

  tb_prediccion <- dfuture[, list(numero_de_cliente)]
  tb_prediccion[, prob_acumulada := 0]

  # entreno el ensemble arbol por arbol, subiendo un submit en cada punto de PARAM$grabar
  for (arbolito in seq(PARAM$num_trees_max)) {
    qty_campos_a_utilizar <- as.integer(length(campos_buenos) * v_feature_fraction)
    campos_random <- sample(campos_buenos, qty_campos_a_utilizar)
    campos_random <- paste(campos_random, collapse= " + ")
    formulita <- paste0("clase_ternaria ~ ", campos_random)

    modelo <- rpart(formulita, data= dtrain, xval= 0, control= rpart_control)
    prediccion <- predict(modelo, dfuture, type= "prob")
    tb_prediccion[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]

    if (!(arbolito %in% PARAM$grabar)) next

    # si este punto puntual ya fue subido en un intento anterior, no lo repito
    #  (igual tuve que re-entrenar los arboles anteriores para llegar hasta aca)
    if (arbolito %in% puntos_hechos) next

    # umbral sobre la SUMA acumulada, equivalente a promedio > 1/40
    umbral_corte <- arbolito / 40
    tb_prediccion[, Predicted := as.numeric(prob_acumulada > umbral_corte)]

    archivo_kaggle <- paste0("KA421_combo_", sprintf("%.3d", combo_id),
      "_arb", sprintf("%.3d", arbolito), ".csv")
    fwrite( tb_prediccion[, list(numero_de_cliente, Predicted)],
      file= archivo_kaggle, sep= "," )

    # subida a Kaggle, el mensaje lleva la combinacion y el arbolito para poder identificarlos despues
    comando <- "kaggle competitions submit"
    competencia <- "-c utn-2026-inicial"
    arch <- paste( "-f", archivo_kaggle)
    mensaje <- paste0("-m 'grid ff=", v_feature_fraction, " cp=", PARAM$rpart$cp,
      " minsplit=", v_minsplit, " minbucket=", v_minbucket, " maxdepth=", v_maxdepth,
      " arbolito=", arbolito, "'")
    linea <- paste( comando, competencia, arch, mensaje)
    salida <- system(linea, intern=TRUE)
    cat(salida)

    tb_grid <- rbindlist(list( tb_grid, data.table(
      combo_id= combo_id,
      feature_fraction= v_feature_fraction,
      cp= PARAM$rpart$cp,
      minsplit= v_minsplit,
      minbucket= v_minbucket,
      maxdepth= v_maxdepth,
      arbolito= arbolito,
      archivo_kaggle= archivo_kaggle
    )))

    # grabo el checkpoint despues de cada submit, para poder retomar
    #  sin volver a gastar submits ya usados
    fwrite(tb_grid, file= archivo_grid, sep= "\t")
  }
}
}
}
}

### Resultado: combinaciones corridas (revisar puntajes en Kaggle > My Submissions)

In [ ]:
tb_grid

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")